# Lecture 18 — pandas: Asking Questions and Adding Columns
### EES 3350/5350 · Python in Earth Science · Friday, October 16, 2026

On Wednesday you met the DataFrame: columns by **name**, rows by **position** (`.iloc`) or **label** (`.loc`), and dates as labels. Today you'll do two things you already know how to do in NumPy, the pandas way:

1. **ask questions with masks**, and get back whole rows, not just one array,
2. **add new columns** to a table, and rename or remove old ones.

**Learning objectives for today**
- build a mask from a column, count it, and use it to filter **rows** of a DataFrame
- combine conditions with `&`, `|`, and `~` (same rules as Lecture 15)
- select rows by mask **and** a column by name in one step with `.loc[mask, "col"]`
- add a column calculated from other columns, and `.rename()` or `.drop()` columns

---
## Part 0 : get today's notebook

```
cd ~/course-materials
git pull
cp week08/18_pandas_masks_columns.ipynb ~/LastName_EES3350/labs/week08/18_pandas_masks_columns_LastName.ipynb
```

Then load the data with Wednesday's shortcut. We keep the columns we need and make our own `.copy()` (Lecture 15), because today we'll be **adding** columns.

In [1]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../../course-materials/week08/nashville_ghcnd_2025.csv", parse_dates=["DATE"], index_col="DATE")

weather = raw[["TMAX", "TMIN", "PRCP", "SNOW", "AWND"]].copy()

weather.head()

,TMAX,TMIN,PRCP,SNOW,AWND
DATE,,,,,
2025-01-01,44,28,0.00,0.0,7.38
2025-01-02,49,25,0.00,0.0,3.13
2025-01-03,44,30,0.00,0.0,8.28
2025-01-04,43,29,0.00,0.0,4.70
2025-01-05,56,34,1.19,0.0,8.05


Five columns: daily high, daily low, precipitation, snowfall, and average wind speed, with the date as the label for every row.

*Data: NOAA NCEI GHCN-Daily, station USW00013897 (Nashville Intl AP, TN), 2025, via [Climate Data Online](https://www.ncdc.noaa.gov/cdo-web/), the same file as Lecture 17.*

---
## Part 1 : masks in pandas

In Lecture 15 you asked "on which days was something true?" with a **mask**. pandas uses the exact same idea, in three steps.

#### Step 1: make the mask
A comparison on a column checks every day and gives back a **boolean Series**: `True`/`False`, one per day, **labeled by date**.

In [2]:
hot = weather["TMAX"] >= 90

hot.loc["2025-06-01":"2025-06-07"]

DATE
2025-06-01    False
2025-06-02    False
2025-06-03     True
2025-06-04    False
2025-06-05    False
2025-06-06    False
2025-06-07    False
Name: TMAX, dtype: bool

#### Step 2: count it
`True` counts as 1 and `False` as 0, so `.sum()` counts the hot days, the same as `np.sum(mask)`:

In [3]:
print(f"Days at 90 F or hotter: {hot.sum()}")

Days at 90 F or hotter: 79


79: the same answer as Lecture 15's `np.sum(tmax >= 90)`.

#### Step 3: filter with it
Put the mask inside square brackets on the **DataFrame**:

In [4]:
hot_days = weather[hot]

print(hot_days.shape)
hot_days.head()

(79, 5)


,TMAX,TMIN,PRCP,SNOW,AWND
DATE,,,,,
2025-06-03,90,67,0.00,0.0,7.61
2025-06-12,90,68,0.00,0.0,2.68
2025-06-15,90,70,0.03,0.0,4.92
2025-06-18,90,74,0.00,0.0,11.18
2025-06-20,91,69,0.00,0.0,3.36


#### What just happened?
`weather[hot]` kept the **79 rows** where `hot` is `True`, and it kept **every column** for those rows: the high, the low, the precipitation, the snow, and the wind, all still lined up with their dates.

Compare that with NumPy, where every column was its own array:

|**question**|**NumPy (Lecture 15)**|**pandas (today)**|
|:--|:--|:--|
|which days were hot?|`tmax >= 90`|`weather["TMAX"] >= 90`|
|how many?|`np.sum(tmax >= 90)`|`(weather["TMAX"] >= 90).sum()`|
|the highs on those days|`tmax[tmax >= 90]`|`weather["TMAX"][weather["TMAX"] >= 90]`|
|**everything** on those days|`tmax[mask]`, `tmin[mask]`, `prcp[mask]`, ... one at a time|`weather[weather["TMAX"] >= 90]`, one table|

In one line, the usual pattern is:

In [5]:
weather[weather["PRCP"] >= 1]

,TMAX,TMIN,PRCP,SNOW,AWND
DATE,,,,,
2025-01-05,56,34,1.19,0.0,8.05
2025-02-11,44,38,1.67,0.0,6.26
2025-02-15,68,43,2.90,0.0,12.75
2025-03-15,71,58,3.75,0.0,14.32
2025-04-03,80,59,3.86,0.0,8.28
2025-04-05,83,59,2.12,0.0,9.62
2025-05-20,83,66,1.62,0.0,NaN
2025-05-25,79,59,1.26,0.0,6.93
2025-08-13,90,73,2.27,0.0,4.70


Read it out loud: **"the rows of `weather` where `PRCP` is at least 1."** That's the 12 days with an inch or more of rain from Lecture 15, and now you can see their dates and everything else about them.

#### Combining conditions: same rules as Lecture 15
`&` (both), `|` (either), `~` (not), with **each condition in its own parentheses**:

In [6]:
hot_and_dry = weather[(weather["TMAX"] >= 90) & (weather["PRCP"] == 0)]

print(f"Hot, dry days: {len(hot_and_dry)}")

Hot, dry days: 56


<div class="alert alert-danger">

**Common Error:** the Lecture 15 errors are back, with the same fixes:
- `(wx["TMAX"] >= 90) and (wx["PRCP"] == 0)` → `ValueError: The truth value of a Series is ambiguous`. Use `&`, not `and`.
- `wx["TMAX"] >= 90 & wx["PRCP"] == 0` → `TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]`. A confusing message! Python did `90 & wx["PRCP"]` first. Add the parentheses.

pandas says "**Series** is ambiguous" where NumPy said "**array**," but it's the same mistake.

</div>

#### Rows by mask, column by name: `.loc[mask, "col"]`
Often you want the rows from a mask but only **one** column. `.loc` takes a row selector and a column selector, separated by a comma, and a mask counts as a row selector:

In [7]:
print(f"Average wind on hot days: {weather.loc[hot, 'AWND'].mean():.1f} mph")
print(f"Average wind, all year:   {weather['AWND'].mean():.1f} mph")

Average wind on hot days: 4.6 mph
Average wind, all year:   6.6 mph


<div class="alert alert-warning">

Hot days in Nashville were noticeably **calmer** than the year as a whole. 

</div>

|**you write**|**read it as**|**you get**|
|:--|:--|:--|
|`wx[mask]`|the rows where `mask` is `True`|a DataFrame, all columns|
|`wx.loc[mask, "AWND"]`|those rows, the `AWND` column|a Series|
|`wx.loc[mask, ["TMAX", "AWND"]]`|those rows, two columns|a DataFrame|
|`wx.loc["2025-07", "TMAX"]`|July (a label), the `TMAX` column|a Series (Wednesday)|

<div class="alert alert-info">

### Exercise 1 : asking the year questions

1. How many days had **any** precipitation (`PRCP` greater than 0) **and** a low at or below 32 °F? These are the days that *could* have had freezing rain or snow.
2. Make `big_rain`, the rows with **1 inch or more** of precipitation. What was the total precipitation on those days? (Lecture 15 answer: 24.66 in.)
3. Use `.loc` with a **mask and a column name** to find the average **high** on days with **1 inch or more** of precipitation. Is it warmer or cooler than the year's average high?
4. How many **summer** days (June 1 through August 31) reached **90 °F**? Hint: first make `summer` with `.loc` and dates (Wednesday), then use a mask on `summer`.

</div>

In [8]:
# your code here
# 1
cold_wet = (weather["PRCP"] > 0) & (weather["TMIN"] <= 32)
print(f"Cold, wet days: {cold_wet.sum()}")

# 2
big_rain = weather[weather["PRCP"] > 1]
print(f"Rain on 1-inch days: {big_rain["PRCP"].sum():.2f} in")

# 3
print(f"Average high on 1-inch days: {weather.loc[weather["PRCP"] >= 1, "TMAX"].mean():.1f} F") 
# 4
summer = weather.loc["2025-06-01":"2025-08-31"]
print(f"Summer 90 F days: {(summer["TMAX"] >= 90).sum()}")

Cold, wet days: 16
Rain on 1-inch days: 24.66 in
Average high on 1-inch days: 71.2 F
Summer 90 F days: 65


<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
cold_wet = (weather["PRCP"] > 0) & (wx["TMIN"] <= 32)
print(f"Cold, wet days: {cold_wet.sum()}")

# 2
big_rain = weather[weather["PRCP"] >= 1]
print(f"Rain on 1-inch days: {big_rain['PRCP'].sum():.2f} in")

# 3
print(f"Average high on 1-inch days: {weather.loc[weather['PRCP'] >= 1, 'TMAX'].mean():.1f} F")
print(f"Average high, all year:      {weather['TMAX'].mean():.1f} F")

# 4
summer = weather.loc["2025-06-01":"2025-08-31"]
print(f"Summer 90 F days: {(summer['TMAX'] >= 90).sum()}")
```
---
```
Cold, wet days: 16
Rain on 1-inch days: 24.66 in
Average high on 1-inch days: 71.2 F
Average high, all year:      73.3 F
Summer 90 F days: 65
```

For #1: 16, the same number as the ice-storm aside in Lecture 15.

For #3: a little cooler, 71.2 °F vs. 73.3 °F. Heavy-rain days are cloudy days, and many of them came in the cooler months.

For #4: 65 of the year's 79 hot days fell in meteorological summer. In Lecture 15 we found the other 14 were in September.

Notice in #3 that `.loc[mask, "TMAX"]` and `big_rain["TMAX"]` give the same thing: a mask is just another way to pick rows.

</details>

### An answer about using masks in their Series vs in their DataFrame form.

In [9]:
print(f"Average high on 1-inch days: {weather.loc[weather["PRCP"] >= 1, "TMAX"].mean():.1f} F") 

Average high on 1-inch days: 71.2 F


Okay so the reason you can't just REPLACE `weather.loc[weather["PRCP"] >= 1` with `big_rain` is because of the formatting with `"TMAX"`. 

Because you're using `[weather["PRCP"] >= 1` as the "row" and `"TMAX"` as the column, when you replace the former with `big_rain` you get an error that says "Cannot index with multidimensional key" because that key (`big_rain`) is a DataFrame. 

In [ ]:
print(f"Average high on 1-inch days: {weather.loc[big_rain, "TMAX"].mean():.1f} F") 

In [11]:
type(big_rain)

pandas.core.frame.DataFrame

In that syntax, it's looking for a Series (a 1D array), which is what our mask `weather["PRCP"] >= 1` is.

In [12]:
type(weather["PRCP"] >= 1)

pandas.core.series.Series

To use `big_rain`, you just need to change the syntax to look at `"TMAX"` within the DataFrame `big_rain`.

In [13]:
print(f"Average high on 1-inch days: {big_rain["TMAX"].mean():.1f} F") 

Average high on 1-inch days: 71.2 F


---
## Part 2 : adding, renaming, and removing columns

#### New columns from math
On Wednesday you noticed that NOAA's `TAVG` column was empty for this station, and in Lecture 14 we built our own daily average as `(tmax + tmin) / 2`. In pandas, assigning to a **new** column name adds that column to the table:

In [16]:
weather["TAVG"] = (weather["TMAX"] + weather["TMIN"]) / 2

weather.head(3)

,TMAX,TMIN,PRCP,SNOW,AWND,RANGE_F,TAVG
DATE,,,,,,,
2025-01-01,44,28,0.0,0.0,7.38,16,36.0
2025-01-02,49,25,0.0,0.0,3.13,24,37.0
2025-01-03,44,30,0.0,0.0,8.28,14,37.0


#### What just happened?
- On the right, `weather["TMAX"] + weather["TMIN"]` is element-wise math, day by day, exactly like arrays. pandas lines the days up by their **labels**.
- On the left, `weather["TAVG"] = ...` uses a name that doesn't exist yet, so pandas **adds** a column with that name at the end of the table.

The same pattern works for unit conversions:

In [17]:
weather["TMAX_C"] = (weather["TMAX"] - 32) * 5 / 9
weather["PRCP_MM"] = weather["PRCP"] * 25.4

weather.head(3)

,TMAX,TMIN,PRCP,SNOW,AWND,RANGE_F,TAVG,TMAX_C,PRCP_MM
DATE,,,,,,,,,
2025-01-01,44,28,0.0,0.0,7.38,16,36.0,6.666667,0.0
2025-01-02,49,25,0.0,0.0,3.13,24,37.0,9.444444,0.0
2025-01-03,44,30,0.0,0.0,8.28,14,37.0,6.666667,0.0


<div class="alert alert-success">

**Tip: name new columns with their units** (`TMAX_C`, `PRCP_MM`), so nobody, including you in two weeks, has to guess. If you write to a name that **already** exists (`wx["TMAX"] = ...`), pandas **replaces** that column. That's how raw data gets lost, so give converted values a new name instead.

</div>

#### Renaming columns
NOAA's codes are short, but `PRCP` and `AWND` aren't very readable. `.rename()` takes a **dictionary** (Lecture 05) of `{"old name": "new name"}` pairs:

In [18]:
weather = weather.rename(columns={"PRCP": "PRCP_IN", "AWND": "WIND_MPH"})

weather.head()

,TMAX,TMIN,PRCP_IN,SNOW,WIND_MPH,RANGE_F,TAVG,TMAX_C,PRCP_MM
DATE,,,,,,,,,
2025-01-01,44,28,0.00,0.0,7.38,16,36.0,6.666667,0.000
2025-01-02,49,25,0.00,0.0,3.13,24,37.0,9.444444,0.000
2025-01-03,44,30,0.00,0.0,8.28,14,37.0,6.666667,0.000
2025-01-04,43,29,0.00,0.0,4.70,14,36.0,6.111111,0.000
2025-01-05,56,34,1.19,0.0,8.05,22,45.0,13.333333,30.226


Only the columns you list change; the rest keep their names. Notice the `weather = ...` at the start: `.rename()` gives back a **new** DataFrame, and the old one stays the same unless you save the result.

#### Removing columns
`.drop()` takes a list of column names:

In [19]:
weather = weather.drop(columns=["SNOW"])

weather.head()

,TMAX,TMIN,PRCP_IN,WIND_MPH,RANGE_F,TAVG,TMAX_C,PRCP_MM
DATE,,,,,,,,
2025-01-01,44,28,0.00,7.38,16,36.0,6.666667,0.000
2025-01-02,49,25,0.00,3.13,24,37.0,9.444444,0.000
2025-01-03,44,30,0.00,8.28,14,37.0,6.666667,0.000
2025-01-04,43,29,0.00,4.70,14,36.0,6.111111,0.000
2025-01-05,56,34,1.19,8.05,22,45.0,13.333333,30.226


<div class="alert alert-danger">

**Common Error:** forgetting `weather = ...` in front of `.rename()` or `.drop()`. The cell runs with no error, prints the right-looking table, and then `weather` is **unchanged** the next time you use it. If a rename "didn't stick," check for the missing `weather =`.

</div>

<div class="alert alert-info">

### Exercise 2 : from scratch

No blanks this time.

1. Add a column called `RANGE`: the daily high minus the daily low.
2. On which date was the daily range the **biggest**, and how big was it? (Wednesday's `.idxmax()` gives you the date.)
3. Rename `RANGE` to `RANGE_F`, since it's in °F.
4. Use a mask to find the average daily range on **dry** days and on days with **any** precipitation. Which has the bigger range? Why might that be?

</div>

In [20]:
# your code here
# 1
weather["RANGE"] = weather["TMAX"] - weather["TMIN"]

# 2
a = weather["RANGE"].idxmax()
print(f"Biggest range: {weather["RANGE"].max()} F on {a:%B %d}")

# 3
weather = weather.rename(columns={"RANGE":"RANGE_F"})
weather.head()

# 4
dry = weather["PRCP_IN"] == 0
print(f"Dry days: {weather.loc[dry, 'RANGE_F'].mean():.1f} F")
print(f"Wet days: {weather.loc[~dry, 'RANGE_F'].mean():.1f} F")

Biggest range: 42 F on February 26


TypeError: unsupported format string passed to Series.__format__

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
weather["RANGE"] = weather["TMAX"] - weather["TMIN"]

# 2
day = weather["RANGE"].idxmax()
print(f"Biggest range: {weather['RANGE'].max()} F on {day:%B %d}")

# 3
weather = weather.rename(columns={"RANGE": "RANGE_F"})

# 4
dry = weather["PRCP_IN"] == 0
print(f"Dry days: {weather.loc[dry, 'RANGE_F'].mean():.1f} F")
print(f"Wet days: {weather.loc[~dry, 'RANGE_F'].mean():.1f} F")
```
---
```
Biggest range: 42 F on February 26
Dry days: 22.9 F
Wet days: 17.3 F
```

For #2: February 26 (80 °F to 38 °F), the same day as Lecture 15's `.argmax()`, but no `month[i]` and `day[i]` needed.

For #4: dry days have the bigger range. Clouds and rain block sunshine during the day and trap heat at night, so wet days don't heat up as much or cool down as much. Notice `~dry` for "not dry": since `PRCP_IN` can't be negative, that's every day with some precipitation.

</details>

### Looking back at our earlier answer about using masks in their Series vs in their DataFrame form.

Here, we were able to use `dry` in the line `print(f"Dry days: {weather.loc[dry, 'RANGE_F'].mean():.1f} F")` because this is the mask as a **Series**


In [14]:
# figure oout what data type the mask is
type(dry)

NameError: name 'dry' is not defined

Earlier, `big_rain` wasn't a **Series**, it was a **DataFrame**

In [ ]:
# looking back at what data type `big_rain` was
type(big_rain)

#### if you have not applied the mask to a DataFrame and it's still an array of booleans (a Series), then you use:
```python
dataframe.loc[series_mask, "column name"].mean()
```
#### if you've already applied the mask to a DataFrame, you end up with a DataFrame and need to use this syntax:
```python
dataframe_masked["column name"].mean()
```

---
## Part 3 : a preview of Monday

Dates in the index know their own parts. `.month` pulls the month number out of every date, and it can become a column:

In [ ]:
weather["MONTH"] = weather.index.month

weather[["TMAX", "MONTH"]].head(3)

Now a mask can pick out any month:

In [ ]:
print(f"July average high: {weather[weather['MONTH'] == 7]['TMAX'].mean():.1f} F")

That's Lecture 15's 94.0 °F again. But what if you want the average high for **every** month? You'd write that line 12 times, or put it in a loop over `range(1, 13)`.

On Monday, you'll learn a way to do all 12 at once, in one line. It's called `groupby`, and it's the most useful thing in pandas.

<div class="alert alert-custom" style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px;">

### Bonus: degree days, the pandas way

In the Lecture 14 bonus, you computed **cooling degree days** (how far each day's average temperature is **above** 65 °F, or 0 if it isn't) and **heating degree days** (how far it's **below**). Redo it with new columns:

1. Add `CDD` and `HDD` columns to `wx`, using `TAVG` and `np.maximum()` from Lecture 14. (`np.maximum()` works on a Series too.)
2. What were the year's totals? (Lecture 14 answers: 2,197 and about 3,034.)
3. Use a mask on `MONTH` to find which had more **cooling** degree days: **June** or **September**?

</div>

In [ ]:
# your code here

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
weather["CDD"] = np.maximum(weather["TAVG"] - 65, 0)
weather["HDD"] = np.maximum(65 - weather["TAVG"], 0)

# 2
print(f"Cooling degree days: {weather['CDD'].sum():.0f}")
print(f"Heating degree days: {weather['HDD'].sum():.0f}")

# 3
print(f"June:      {weather.loc[weather['MONTH'] == 6, 'CDD'].sum():.0f}")
print(f"September: {weather.loc[weather['MONTH'] == 9, 'CDD'].sum():.0f}")
```
---
```
Cooling degree days: 2197
Heating degree days: 3034
June:      438
September: 320
```

For #3: June had more (438 vs. 320), even though September had 14 days at 90 °F or above. September's nights were cooler, so its daily averages ran lower.

NumPy functions like `np.maximum()` accept a Series and hand back a Series, with the dates still attached. That's what "pandas is built on NumPy" looks like in practice.

</details>

---
## A useful table

|**NumPy (Lecture 15)**|**pandas (today)**|
|:--|:--|
|`mask = tmax >= 90`|`mask = weather["TMAX"] >= 90`|
|`np.sum(mask)`|`mask.sum()`|
|`tmax[mask]`, then `tmin[mask]`, then ...|`weather[mask]`: every column at once|
|`(a >= 90) & (b == 0)`|`(weather["A"] >= 90) & (weather["B"] == 0)`: same rules|
|*(n/a)*|`weather.loc[mask, "AWND"]`: rows by mask, one column by name|
|`tavg = (tmax + tmin) / 2`, a new array|`weather["TAVG"] = (weather["TMAX"] + weather["TMIN"]) / 2`, a new column in the table|
|*(n/a)*|`weather = weather.rename(columns={"old": "new"})`|
|*(n/a)*|`weather = weather.drop(columns=["SNOW"])`|
|`month = data[:, 0]`|`weather["MONTH"] = weather.index.month`|